# Predictive Anayltics: Support Vector Machines with Regression

Task - Approach for SVM:
- Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
- How good is your model? Evaluate your model’s performance and comment on its shortfalls.
- Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
- How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

We used the GPU to train this model. In case the model shoulde be trained on the CPU. Change USE_GPU to false.

In [1]:
USE_GPU = False
from run_config import PATHS, MODELS_DIR

In [2]:
if USE_GPU:
    %load_ext cuml.accel
from run_config import PATHS

In [3]:
if USE_GPU:
    import os
    os.environ["LD_LIBRARY_PATH"] = "/mnt/c/Users/bkran/Documents/AAA/Group-3-AAA/.venv/lib64/python3.12/site-packages/nvidia/cuda_runtime/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

    import cuml
    print(cuml.__version__)

In [4]:
TRAIN_SAMPLE = 70_000
GRID_SAMPLE = 70_000 # for running on a subset to get good parameter and the use the whole set validation set over GRID_SEARCH use only GRID_SEARCH rows of data due to runtime issues, for grid search
SPATIAL_UNIT = "HEXAGON" # HEXAGON
SPATIAL_ENCODING = "latlong" # options: embedding, latlong
TIME_UNIT = "1H" # options: 1H, 4H, 24H
H3_RES = "7" # options 7,8

In [5]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [6]:
if TIME_UNIT == "24H":
    N_COMP = [10, 30]
else:
    N_COMP = [100, 300]

In [7]:
import pandas as pd
import numpy as np
#import matplotlib as plt
import datetime
if USE_GPU:
    from cuml import SVR
    from cuml import LinearSVR
else:
    from sklearn.svm import SVR 
    from sklearn.svm import LinearSVR
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils import resample
from sklearn.model_selection import GridSearchCV
#from sklearn.model_selection import RandomizedSearchCV 
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.kernel_approximation import Nystroem
from joblib import load, dump
import h3
from joblib import Memory


## Preparations

In [8]:
INPUT = PATHS.train_test_dir

In [9]:
DATA_PATH_TRAIN = INPUT / F"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_TRAIN.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_TEST.parquet"

In [10]:
MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [11]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [12]:
if len(train_df) > TRAIN_SAMPLE:
   train_df = train_df.sample(n=TRAIN_SAMPLE, random_state=42)

In [13]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2026-04-17 04:00:00,4,5,4,1.0,6.123234e-17,-0.433884,-0.900969,0.866025,0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2026-04-13 03:00:00,4,1,3,1.0,6.123234e-17,0.000000,1.000000,0.707107,0.707107,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2026-04-17 10:00:00,4,5,10,1.0,6.123234e-17,-0.433884,-0.900969,0.500000,-0.866025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2026-04-15 07:00:00,4,3,7,1.0,6.123234e-17,0.974928,-0.222521,0.965926,-0.258819,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2026-04-17 04:00:00,4,5,4,1.0,6.123234e-17,-0.433884,-0.900969,0.866025,0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [14]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [15]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [16]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: hexa")
    for df in (train_df, val_df, test_df):
        df["lat"], df["lon"] = zip(*df["h3_cell"].map(h3.cell_to_latlng))

Encoding: latlong and Unit: hexa


In [17]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else: 
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

Create y

In [18]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Grid Search

In [19]:
model = SVR()

In [20]:
memory = Memory(location="/tmp/sklearn_cache", verbose=0)

# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVR(max_iter=100_000,tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()),
    ('svm', SVR(max_iter=50_000, tol=1e-2))
], memory=memory)

# regressor
ttr_linear = TransformedTargetRegressor(regressor=pipe_linear, transformer=StandardScaler())
ttr_kernel = TransformedTargetRegressor(regressor=pipe_kernel, transformer=StandardScaler())

# parameter for each kernel # excluded C=100, C=10, 0,001 excluded via testing due to convergance issues
param_grid_linear = {
    "regressor__svm__C": [1, 10, 30, 100], # 4h: , 1h: 24h: 
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3],
}

param_grid_rbf_sigmoid = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["rbf", "sigmoid"],
    "regressor__feature_map__gamma": [0.001, 0.01, 0.1],
    "regressor__feature_map__n_components": N_COMP, # 24H: 10, 30; 1H/4H: 100, 300
}

param_grid_poly = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["poly"],
    "regressor__feature_map__degree": [3, 4],
    "regressor__feature_map__gamma": [0.001, 0.01, 0.1],
    "regressor__feature_map__n_components": N_COMP, # 24H: 10, 30; 1H/4H: 100, 300
}

grids = {}
configs = [
    ("linear", ttr_linear, param_grid_linear),
    ("rbf_sigmoid", ttr_kernel, param_grid_rbf_sigmoid),
    ("poly", ttr_kernel, param_grid_poly),
]

# doing gridsearch on all
for name, pipe, grid in configs:
    search = HalvingGridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=2, # changed to 2 due to runtime issues
        scoring="r2",
        n_jobs=1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.026155371400657157 best params: {'regressor__svm__C': 30, 'regressor__svm__epsilon': 0.3}


/Users/lennartjekel/dev/git/Group-3-AAA/.venv/lib/python3.10/site-packages/sklearn/kernel_approximation.py:1020: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
/Users/lennartjekel/dev/git/Group-3-AAA/.venv/lib/python3.10/site-packages/sklearn/kernel_approximation.py:1020: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
/Users/lennartjekel/dev/git/Group-3-AAA/.venv/lib/python3.10/site-packages/sklearn/kernel_approximation.py:1020: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
/Users/lennartjekel/dev/git/Group-3-AAA/.venv/lib/python3.10/site-packages/sklearn/kernel_approximation.py:1020: UserWarning: n_components > n

rbf_sigmoid best score: 0.24457286528015976 best params: {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.05}


/Users/lennartjekel/dev/git/Group-3-AAA/.venv/lib/python3.10/site-packages/sklearn/kernel_approximation.py:1020: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
/Users/lennartjekel/dev/git/Group-3-AAA/.venv/lib/python3.10/site-packages/sklearn/kernel_approximation.py:1020: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
/Users/lennartjekel/dev/git/Group-3-AAA/.venv/lib/python3.10/site-packages/sklearn/kernel_approximation.py:1020: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
/Users/lennartjekel/dev/git/Group-3-AAA/.venv/lib/python3.10/site-packages/sklearn/kernel_approximation.py:1020: UserWarning: n_components > n

poly best score: 0.21223431681477933 best params: {'regressor__feature_map__degree': 4, 'regressor__feature_map__gamma': 0.1, 'regressor__feature_map__kernel': 'poly', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 1, 'regressor__svm__epsilon': 0.1}
Overall best: rbf_sigmoid {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.05}


In [21]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.05}
Best CV score: 0.24457286528015976


### Train Model

In [22]:
best_model = grid_search.best_estimator_

In [23]:
# Train SVR 
best_model.fit(X_train, y_train)

/Users/lennartjekel/dev/git/Group-3-AAA/.venv/lib/python3.10/site-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


,regressor,"Pipeline(memo..., tol=0.01))])"
,transformer,StandardScaler()
,func,None
,inverse_func,None
,check_inverse,True
,copy,True
,with_mean,True
,with_std,True
,kernel,'rbf'
,gamma,0.01
,coef0,None


In [24]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [25]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))


result = {
    "MAE": mean_absolute_error(y_test, y_pred),
    "MSE": mean_squared_error(y_test, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2 Score": r2_score(y_test, y_pred),
}

MAE: 1.8497706712230817
MSE: 40.01922752972493
RMSE: 6.326075207403476
R2 Score: 0.8772422135380272


In [26]:
df = pd.DataFrame({ # did not reorder at any point
    "y_pred": y_pred,
    "y_test": y_test,
    "h3_cell": test_df["h3_cell"].values,
    "date": test_df["datetime_hour"].values,
})
df.to_csv(MODELS_DIR / f"svm/model_{SPATIAL_UNIT}_{H3_RES}_{TIME_UNIT}.csv", index=False)

In [27]:
pd.DataFrame([result]).to_csv(MODELS_DIR / f"svm/result_{SPATIAL_UNIT}_{H3_RES}_{TIME_UNIT}.csv", index=False)

In [28]:
dump(best_model, MODELS_DIR / f"svm/model_{SPATIAL_UNIT}_{H3_RES}_{TIME_UNIT}_svr.joblib")
dump(grid_search, MODELS_DIR / f"svm/grid_{SPATIAL_UNIT}_{H3_RES}_{TIME_UNIT}_svr.joblib")

['/Users/lennartjekel/dev/git/Group-3-AAA/models/sample/svm/grid_HEXAGON_7_1H_svr.joblib']